<a href="https://colab.research.google.com/github/kimjiji8105/class2025Spring/blob/main/Toxic_Comment_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mini Project: Hate Speech and Abusive Comment Detection

# 환경 세팅

In [ ]:
# #처음 실행 후 세션 다시 시작 - plt 한글 깨짐 현상 방지
# !sudo apt-get install -y fonts-nanum       # Nanum 폰트 설치
# !sudo fc-cache -fv                         # 폰트 캐시 갱신
# !rm -rf ~/.cache/matplotlib                 # matplotlib 캐시 삭제

In [ ]:
!pip install -q transformers evaluate accelerate datasets koco peft==0.13.2

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import koco
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from google.colab import drive
from transformers.integrations import WandbCallback


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import evaluate

import joblib
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
#행·열 제한 해제
#pd.set_option('display.max_rows',    None)   # 행 모두
pd.set_option('display.max_columns', None)   # 열 모두

#한글 깨짐 해결
plt.rc('font', family='NanumBarunGothic')       # 설치된 나눔바른고딕 계열
plt.rcParams['axes.unicode_minus'] = False      # 마이너스 깨짐 방지

In [ ]:
#변수 정의
cols = ['category', '문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'target']
sub_cols = ['여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설']

In [ ]:
#Drive 마운트
drive.mount('/content/drive')

In [ ]:
#dir path 지정
output_dir = '/content/drive/MyDrive/Toxic_Comment_Detection/kobert_binary_outputs/'
model_dir = '/content/drive/MyDrive/Toxic_Comment_Detection/models/'

In [ ]:
#output 저장 폴더 생성
folders = [output_dir, model_dir]

for path in folders:
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")

# 데이터 로드 & 구조 확인

[사용 데이터]
- korean_unsmile_dataset : https://github.com/smilegate-ai/korean_unsmile_dataset.git
- hatescore-korean-hate-speech: https://github.com/sgunderscore/hatescore-korean-hate-speech.git

[확인 내용]
- 데이터 오류 여부
- 정상, 부정 여부

## unsmile_df

### 테이블 로드

In [ ]:
datasets = load_dataset('smilegate-ai/kor_unsmile')
print(datasets)

unsmile_df = pd.concat([pd.DataFrame(datasets['train']), pd.DataFrame(datasets['valid'])])

### 구조 확인

In [ ]:
unsmile_df.info()

In [ ]:
#중복 있음 확인
unsmile_df.describe(include='all')

In [ ]:
unsmile_df.isnull().sum()

In [ ]:
unsmile_df.head()

In [ ]:
unsmile_df['clean'].value_counts()

In [ ]:
unsmile_df.loc[:,'여성/가족':'clean'].value_counts().reset_index()

#clean이 1이면서 혐오표현이 1인 경우(오류)는 없지만, 혐오 표현이 2개 이상인 경우는 존재
#우리는 clean 여부만 사용할 것이므로 문제 X

In [ ]:
unsmile_df['labels']

In [ ]:
#우리는 문제성 글을 1, 아닌 경우를 0을 기준으로 함
#따라서, clean이 1-> 0, 0-> 1 처리 필요

In [ ]:
# 중복 제거 - 개인적으론.. 악풀까진 아닌듯.. clean 1로 남기기
unsmile_df[unsmile_df.duplicated('문장', keep=False)]

In [ ]:
unsmile_df.drop_duplicates('문장', keep='last', inplace=True)

## hatescore_df

### 테이블 로드

In [ ]:
# “main” 브랜치의 HateScore.csv 파일에 접근
url = (
    "https://raw.githubusercontent.com/"
    "sgunderscore/hatescore-korean-hate-speech/"
    "main/"                       # ← 여기에 브랜치 이름이 꼭 들어가야 합니다
    "HateScore.csv"
)

In [ ]:
# CSV 파일이기 때문에 sep은 기본 쉼표(,)를 사용
hatescore_df  = pd.read_csv(url, index_col=0)
hatescore_df

### 구조 확인

In [ ]:
hatescore_df.info()

In [ ]:
hatescore_df.describe(include='all')

In [ ]:
hatescore_df.isnull().sum()

In [ ]:
hatescore_df['macrolabel'].value_counts()

In [ ]:
hatescore_df.head()

In [ ]:
hatescore_df['microlabel'].value_counts()

In [ ]:
hatescore_df.loc[:,'macrolabel':'microlabel'].value_counts().reset_index()

In [ ]:
#우리는 문제성 글을 1, 아닌 경우를 0을 기준으로 함
#따라서, clean이 1-> 0, 0-> 1 처리 필요

In [ ]:
hatescore_df[hatescore_df['macrolabel'] == '단순 악플']

In [ ]:
unsmile_df[unsmile_df['악플/욕설'] == 1]

In [ ]:
# hatescore_df 내 단순 악플은 unsmile_df 내 악플/욕설로 대체해도 될듯

# 데이터 전처리

## unsmile_df

In [ ]:
#target 컬럼 추가
unsmile_df.loc[unsmile_df['clean'] == 1, 'target'] = 0
unsmile_df['target'].fillna(1, inplace=True)

#type 변경
unsmile_df['target'] = unsmile_df['target'].astype(int)

#category 컬럼 추가
unsmile_df['category'] = 'unsmile'

In [ ]:
#전처리 정상적으로 된 것 확인
unsmile_df[unsmile_df['clean'] == unsmile_df['target']]

In [ ]:
unsmile_df

## hatescore_df

target 컬럼 추가

In [ ]:
hatescore_df.loc[hatescore_df['macrolabel'] == 'Clean', 'target'] = 0
hatescore_df['target'].fillna(1, inplace=True)

#type 변경
hatescore_df['target'] = hatescore_df['target'].astype(int)

#category 컬럼 추가
hatescore_df['category'] = 'hatescore'

In [ ]:
hatescore_df

더미 변수(dummy variable) 생성

In [ ]:
#구분자를 ', ' 로 정확히 지정
dummies = hatescore_df['microlabel'] \
    .fillna('') \
    .str.get_dummies(sep=', ')

In [ ]:
#원본 DF 에 병합
hatescore_df = pd.concat([hatescore_df, dummies], axis=1)
hatescore_df.head()

In [ ]:
hatescore_df[hatescore_df['microlabel'] == '기타 혐오, 남성, 성소수자, 여성/가족']

컬럼명 변경

In [ ]:
#단순 악플 -> 악플/욕설
hatescore_df.rename(columns={'단순 악플': '악플/욕설'}, inplace=True)

#comment -> 문장
hatescore_df.rename(columns={'comment': '문장'}, inplace=True)
hatescore_df

## 결합

In [ ]:
dataset = pd.concat([unsmile_df[cols], hatescore_df[cols]])
dataset

### 데이터 확인

In [ ]:
dataset.info()

In [ ]:
#문장 중복 없음
dataset.describe(include='all')

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset.head()

In [ ]:
#target : 1 -> 문제O / 0 -> 문제X
#클래스 균형이 잘 맞음
dataset['target'].value_counts()

In [ ]:
#중복 확인 : 있음
dataset[dataset.duplicated('문장', keep=False)]

## EDA

In [ ]:
# 정상 및 혐오 수 분포
sum_df = dataset['target'].value_counts()
sum_df

In [ ]:
#시각화
plt.figure(figsize=(10, 6))
ax = sum_df.plot(kind='bar')

# x축 라벨 회전
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# 축 제목
ax.set_xlabel('구분', fontsize=14)
ax.set_ylabel('개수', fontsize=14)

# 타이틀 (크기 키우기, 아래 여백 패딩 추가)
ax.set_title('정상 및 혐오 수 분포', fontsize=20, pad=20)

plt.tight_layout()
plt.show()

In [ ]:
#혐오 유형 수
sum_df = dataset.loc[:, sub_cols].sum(axis=0)
sum_df

In [ ]:
#시각화
plt.figure(figsize=(10, 6))
ax = sum_df.plot(kind='bar')

# x축 라벨 회전
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# 축 제목
ax.set_xlabel('종류', fontsize=14)
ax.set_ylabel('개수', fontsize=14)

# 타이틀 (크기 키우기, 아래 여백 패딩 추가)
ax.set_title('혐오 유형 수 분포', fontsize=20, pad=20)

plt.tight_layout()
plt.show()

In [ ]:
#문장별 혐오 유형 수 분포
sum_df = dataset.loc[:, sub_cols].sum(axis=1)
sum_df = sum_df.value_counts()
sum_df

In [ ]:
#시각화
plt.figure(figsize=(10, 6))
ax = sum_df.plot(kind='bar')

# x축 라벨 회전
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# 축 제목
ax.set_xlabel('합계', fontsize=14)
ax.set_ylabel('개수', fontsize=14)

# 타이틀 (크기 키우기, 아래 여백 패딩 추가)
ax.set_title('문장별 혐오 유형 수 분포', fontsize=20, pad=20)

plt.tight_layout()
plt.show()

## 데이터 분리

In [ ]:
# 3) 레이블 컬럼(예: 'labels' 또는 'clean') 선택
X = dataset['문장'].values           # 입력 텍스트
y = dataset['target'].values          # 0 or 1 (깨끗한 문장 여부)  ※ 필요에 맞게 바꾸세요

In [ ]:
# 4) Train / Val / Test 80 : 10 : 10 분리 ─────────────
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

In [ ]:
X_train

In [ ]:
print(f"Train  : {len(X_train)}")
print(f"Valid. : {len(X_val)}")
print(f"Test   : {len(X_test)}")

In [ ]:
print(f"Train  : 1 : {(y_train == 1).sum()} / 0 : {(y_train == 0).sum()}")
print(f"Valid  : 1 : {(y_val == 1).sum()}  / 0 : {(y_val == 0).sum()}")
print(f"Test   : 1 : {(y_test == 1).sum()}  / 0 : {(y_test == 0).sum()}")

# 베이스라인 모델 (TF-IDF + Logistic Regression)
가장 흔히 사용하는 베이스라인 모델을 선택함

In [ ]:
#파이프라인 정의
pipe = make_pipeline(
    TfidfVectorizer(                         # # 토크나이저 정의
        token_pattern=r'(?u)\b[\w가-힣]+\b', # 한글·영문·숫자 단어만 토큰
        ngram_range=(1,2),                   # 1-gram(단어) + 2-gram(연속 두 단어)
        max_features=40000,                  # 가장 빈도 높은 4만개 토큰만
        min_df=2                             # 빈도 2회 미만 토큰 제거 - 노이즈 제거 용도
    ),
    LogisticRegressionCV(                    # # 모델 정의
        cv=5,                                # 5-fold 교차검증
        class_weight='balanced',             # 레이블 빈도에 반비례 가중치
        max_iter=1000,                       # 최대 반복 수
        n_jobs=-1,                           # 병렬 학습
        multi_class='ovr',                   # 이진(vs_rest) 분류 - 기본값임(다중 클래스일 때 대비해 명시)
        solver='lbfgs',                      # 최적화 알고리즘
        random_state=42,                     # 재현성 고정
        scoring='f1'                         # binary F1-score로 C 최적화
    )
)

In [ ]:
#학습
pipe.fit(X_train, y_train)
pred = pipe.predict(X_val)

In [ ]:
#모델 저장
joblib.dump(pipe, model_dir+"baseline_tf_logreg.joblib")

# BERT 파인튜닝

- 사용 모델: monologg/kobert
- https://huggingface.co/monologg/kobert/tree/main

데이터 전처리

*NumPy 배열 → Hugging Face Dataset으로 변환

In [ ]:
def make_hf_dataset(X, y):
    return Dataset.from_dict({"문장": list(X), "labels": list(y)})

raw_ds = DatasetDict({
    "train":      make_hf_dataset(X_train, y_train),
    "validation": make_hf_dataset(X_val,   y_val),
    "test":       make_hf_dataset(X_test,  y_test),
})

토크나이저 로드 & 전처리

In [ ]:
#토크나이저 로드
MODEL_NAME = "monologg/kobert"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [ ]:
#TypeError: save_vocabulary() got an unexpected keyword argument 'filename_prefix'” 오류 해결을 위한 패치
#토크나이저 인스턴스가 생성된 직후, 해당 인스턴스의 save_vocabulary 메서드를 래퍼 함수로 덮어써서 filename_prefix 인자를 무시하도록 Monkey-Patch를 적용

#원본 save_vocabulary 백업
_orig_save_vocab = tokenizer.save_vocabulary

#filename_prefix 인자 무시 래퍼 정의
def _patched_save_vocabulary(save_directory, filename_prefix=None):
    return _orig_save_vocab(save_directory)

#인스턴스 메서드 덮어쓰기
tokenizer.save_vocabulary = _patched_save_vocabulary

print("✅ KoBertTokenizer.save_vocabulary 패치 완료")

In [ ]:
def preprocess(batch):
    return tokenizer(
        batch["문장"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
tokenized_ds = raw_ds.map(preprocess, batched=True)
tokenized_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

모델 로드 (이진 분류)

*base KoBERT 가중치는 로드, 헤드만 랜덤 초기화

In [ ]:
#분류 모델 로드 & LoRA 어댑터 적용
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2                                        # 이진 분류(0,1)
    )

lora_cfg = LoraConfig(
    task_type="SEQ_CLS",                                # Sequence Classification(문장 분류) 전용 어댑터 - 분류용 head(CLS 토큰 출력) 위주로 최적화된 구조로 LoRA를 결합하게 됨
    r=8,                                                # LoRA rank - 차원 축소 크기(rank)
    lora_alpha=16,                                      # ΔW에 곱해지는 스케일링 계수
    target_modules=["query", "key", "value"],           # Transformer 어텐션 레이어 중 Q/K/V 프로젝션에만 LoRA를 적용
    inference_mode=False                                # 학습 모드 - LoRA 가중치가 업데이트
)
model = get_peft_model(base_model, lora_cfg)

In [ ]:
args = TrainingArguments(
    output_dir=output_dir,

    # ─── 평가·저장 주기 설정 ─────────────────────────────
    eval_strategy   = "epoch",   # epoch 단위 평가
    save_strategy   = "epoch",   # epoch 단위 체크포인트 저장
    logging_strategy= "steps",   # 로깅 주기 제어
    logging_steps   = 100,       # 100 스텝마다 로그

    # ─── 학습 설정 ──────────────────────────────────
    num_train_epochs           = 3,
    per_device_train_batch_size= 16,
    per_device_eval_batch_size = 16,
    learning_rate              = 1e-4,
    weight_decay               = 0.01,

    # ─── 최고 모델 저장 설정 ─────────────────────────────
    load_best_model_at_end = True,     # 검증 지표 기반 최적 모델 로드
    metric_for_best_model  = "f1",     # compute_metrics 가 반환하는 키 이름
    greater_is_better      = True,     # 지표 값이 클수록 좋음

    # ─── 기타 ────────────────────────────────────
    fp16=True,      # GPU용 fp16 활성화
    seed = 42,
    report_to="none",       # W&B, tensorboard 등 외부 로깅 전부 끔
)

평가 지표 함수 (sklearn 사용)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall":    recall_score(labels, preds),
        "f1":        f1_score(labels, preds)
    }

In [ ]:
#Trainer 구성
trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = tokenized_ds["train"],
    eval_dataset    = tokenized_ds["validation"],
    compute_metrics = compute_metrics,
    tokenizer       = None
)

학습 & 평가

In [ ]:
trainer.train()   # 약 15분 소요

In [ ]:
#모델 및 토크나이저 저장
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

# 테스트 평가 & 성능 지표

## TF-IDF + Logistic Regression

In [ ]:
#모델 불러오기
pipe = joblib.load(model_dir+"baseline_tf_logreg.joblib")

In [ ]:
#테스트셋 평가
baseline_preds = pipe.predict(X_test)

In [ ]:
#성능 확인
print(classification_report(
    y_true=y_val,
    y_pred=baseline_preds,
    digits=4
))

In [ ]:
#confusion_matrix
cm_base = confusion_matrix(y_test, baseline_preds)
labels = ['Non-toxic (0)', 'Toxic (1)']

#히트맵
plt.figure()
plt.imshow(cm_base, cmap='Blues')
plt.title('Baseline Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks([0, 1], labels)
plt.yticks([0, 1], labels)
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_base[i, j], ha='center', va='center')
plt.show()

## KoBERT 기반 모델

In [ ]:
#모델 불러오기
model = AutoModelForSequenceClassification.from_pretrained(
    model_dir,
    trust_remote_code=True
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_dir,trust_remote_code=True)

# 패치 코드 실행
_orig = tokenizer.save_vocabulary
def _patched(save_directory, filename_prefix=None):
    return _orig(save_directory)
tokenizer.save_vocabulary = _patched

In [ ]:
trainer = Trainer(
  model=model,
  tokenizer=tokenizer
)

In [ ]:
#WandbCallback 제거 - 연동 안하려고 제거
trainer.remove_callback(WandbCallback)

In [ ]:
#테스트셋 평가
bert_preds = trainer.predict(tokenized_ds["test"])

In [ ]:
#성능 확인
bert_preds = np.argmax(pred_out.predictions, axis=-1)
print(classification_report(tokenized_ds["test"]["labels"], preds, digits=4))

In [ ]:
#성능 확인
print(classification_report(
    y_true=y_val,
    y_pred=baseline_preds,
    digits=4
))

In [ ]:
#confusion_matrix
cm_bert = confusion_matrix(y_test, bert_preds)

#히트맵
plt.figure()
plt.imshow(cm_bert, cmap='Blues')
plt.title('BERT Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks([0, 1], labels)
plt.yticks([0, 1], labels)
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm_bert[i, j], ha='center', va='center')
plt.show()

# 결과 분석

In [ ]:
#결과 df 만들기
df_results = pd.DataFrame({
    "문장": X_test,
    "baseline_pred": baseline_preds,
    "bert_pred": bert_preds
})

#구체적인 분석을 위한 원 테이블과 결합
#'문장'은 unique한 값이므로 key 로 활용 가능 - 해당 데이터에서는
df_results = df_results.merge(dataset, on='문장', how='left')
df_results = df_results[['문장', 'target', 'baseline_pred', 'bert_pred', 'category', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설']]

In [ ]:
#문제 없이 결합된 것 확인
print(len(baseline_preds))
print(len(bert_preds))
df_results.info()

In [ ]:
# Baseline 오분류
base_errors = df_results[df_results["baseline_pred"] != df_results["target"]]
# Baseline 정답
base_correct = df_results[df_results["baseline_pred"] == df_results["target"]]


# BERT 오분류
bert_errors = df_results[df_results["bert_pred"] != df_results["target"]]
# BERT 정답
bert_correct = df_results[df_results["bert_pred"] == df_results["target"]]

##“어떤 유형의 토픽”에서 오답이 많았는가?

**Base 모델**  
- 오류 유형이 BERT에 비해 비교적 획일적임  
- 모델은 문제성으로 판단했지만, 실제 정답은 정상인 경우 (496건)가 모델은 정상으로 판단했지만, 실제 문제성인 경우 (56건) 에 비해 많았음
- 오답이 많은 토픽 유형: 악플/욕설(21건), 성소수자(8건), 여성/가족(6건), 지역(5건) 순  


In [ ]:
#추후.. 순서 고려한 비율로 분석하는게 맞을 듯...
dataset.loc[:,'여성/가족':'악플/욕설'].value_counts().reset_index()

In [ ]:
base_errors['target'].value_counts()

In [ ]:
#오류 유형이 base에 비해 비교적 획일적임
base_errors.loc[:,'여성/가족':'악플/욕설'].value_counts().reset_index()

In [ ]:
#모델은 문제성으로 봤지만, 정답은 정상으로 본 경우 : 496건
base_errors[base_errors['target'] == 0]

In [ ]:
#모델은 정상으로 봤지만, '악플/욕설' 유형의 문제가 있는 경우 : 21건
base_errors[base_errors['악플/욕설'] == 1]


**BERT 모델**  
- 오류 유형이 Base 모델에 비해 비교적 다양함  
- 모델은 문제성으로 판단했지만 실제 정답은 정상인 경우(221건)가, 모델은 정상으로 판단했지만 실제로는 문제성인 경우(181건)보다 다소 많았음.
- 오답이 많은 토픽 유형: 악플/욕설(99건), 여성/가족(18건), 지역(13건), 성소수자(12건) 순  
- Base 모델과 오답 순서 및 분포는 다르나, 주요 유형은 유사함

In [ ]:
bert_errors['target'].value_counts()

In [ ]:
#오류 유형이 base에 비해 다양함
bert_errors.loc[:,'여성/가족':'악플/욕설'].value_counts().reset_index()

In [ ]:
#모델은 문제성으로 봤지만, 정답은 정상으로 본 경우 : 221건
bert_errors[bert_errors['target'] == 0]

In [ ]:
bert_errors[bert_errors['악플/욕설'] == 1]

##“어떤 유형의 토픽”에서 정답이 많았는가?

**Base 모델**  
- 오류 유형이 BERT에 비해 비교적 획일적임  
- 모델은 문제성으로 판단했지만, 실제 정답은 정상인 경우 (496건)가 모델은 정상으로 판단했지만, 실제 문제성인 경우 (56건) 에 비해 많았음
- 오답이 많은 토픽 유형: 악플/욕설(21건), 성소수자(8건), 여성/가족(6건), 지역(5건) 순  


In [ ]:
base_correct['target'].value_counts()

In [ ]:
#오류 유형이 base에 비해 비교적 획일적임
base_correct.loc[:,'여성/가족':'악플/욕설'].value_counts().reset_index()

In [ ]:
base_correct[base_correct['target'] == 0]

In [ ]:
base_correct[base_correct['악플/욕설'] == 1]


**BERT 모델**  
- 오류 유형이 Base 모델에 비해 비교적 다양함  
- 모델은 문제성으로 판단했지만 실제 정답은 정상인 경우(221건)가, 모델은 정상으로 판단했지만 실제로는 문제성인 경우(181건)보다 다소 많았음.
- 오답이 많은 토픽 유형: 악플/욕설(99건), 여성/가족(18건), 지역(13건), 성소수자(12건) 순  
- Base 모델과 오답 순서 및 분포는 다르나, 주요 유형은 유사함

In [ ]:
bert_correct['target'].value_counts()

In [ ]:
#오류 유형이 base에 비해 다양함
bert_correct.loc[:,'여성/가족':'악플/욕설'].value_counts().reset_index()

In [ ]:
#모델은 문제성으로 봤지만, 정답은 정상으로 본 경우 : 221건
bert_correct[bert_correct['target'] == 0]

In [ ]:
bert_correct[bert_correct['악플/욕설'] == 1]